# Anime Recommender · Notebook 3 (期末改版): 評估與比較 — 含 Multimodal

- 對 test 集中每位 user 取 Top-10 推薦
- 計算 Precision@10、Recall@10、NDCG@10
- **(期末新增)** 把 Multimodal 也納入比較
- 與 Popularity baseline 比較
- 產出 `metrics.json` 與 `metrics_comparison.png`,本地 Streamlit 直接讀取顯示

> **Ground truth 定義**:test set 裡使用者「實際給了高分(≥ 8)」的作品。

## Step 1 — 載入資料與模型

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pickle, json
import numpy as np
import pandas as pd
from scipy import sparse

ARTIFACTS_DIR = '/content/drive/MyDrive/anime-recsys/artifacts'

with open(f'{ARTIFACTS_DIR}/mappings.pkl', 'rb') as f:
    mappings = pickle.load(f)
with open(f'{ARTIFACTS_DIR}/user_history.pkl', 'rb') as f:
    user_history = pickle.load(f)

train_df = pd.read_parquet(f'{ARTIFACTS_DIR}/train.parquet')
test_df = pd.read_parquet(f'{ARTIFACTS_DIR}/test.parquet')
anime_meta = pd.read_parquet(f'{ARTIFACTS_DIR}/anime_meta.parquet')

user_id_to_idx = mappings['user_id_to_idx']
anime_id_to_idx = mappings['anime_id_to_idx']
idx_to_anime_id = mappings['idx_to_anime_id']
N_USERS, N_ITEMS = len(user_id_to_idx), len(anime_id_to_idx)
print(f'#users = {N_USERS:,}, #items = {N_ITEMS:,}')

In [ ]:
# 載入所有模型 (期末新增 text_embeddings)
content_sim = sparse.load_npz(f'{ARTIFACTS_DIR}/content_sim.npz')

with open(f'{ARTIFACTS_DIR}/user_cf_model.pkl', 'rb') as f:
    user_cf = pickle.load(f)
with open(f'{ARTIFACTS_DIR}/svd_model.pkl', 'rb') as f:
    svd = pickle.load(f)

text_emb = np.load(f'{ARTIFACTS_DIR}/text_embeddings.npy')
print(f'text_embeddings shape: {text_emb.shape}')
print('All models loaded ✅')

## Step 2 — 建立 test 的 ground truth

對每位 user,把 test 中評分 >= 8 的 anime 視為「真的喜歡」。

In [ ]:
POSITIVE_THRESHOLD = 8

test_pos = test_df[test_df['rating'] >= POSITIVE_THRESHOLD]
gt_dict = test_pos.groupby('user_id')['anime_id'].apply(set).to_dict()
eval_users = [u for u in gt_dict if u in user_id_to_idx and len(gt_dict[u]) >= 1]
rng = np.random.RandomState(42)
if len(eval_users) > 1000:
    eval_users = rng.choice(eval_users, size=1000, replace=False).tolist()
print(f'評估 {len(eval_users)} 位 user')

## Step 3 — 定義評估指標

In [ ]:
def precision_at_k(recommended, relevant, k=10):
    top_k = recommended[:k]
    if not top_k: return 0.0
    return len(set(top_k) & relevant) / k

def recall_at_k(recommended, relevant, k=10):
    top_k = recommended[:k]
    if not relevant: return 0.0
    return len(set(top_k) & relevant) / len(relevant)

def ndcg_at_k(recommended, relevant, k=10):
    top_k = recommended[:k]
    dcg = 0.0
    for i, item in enumerate(top_k):
        if item in relevant:
            dcg += 1.0 / np.log2(i + 2)
    ideal_n = min(len(relevant), k)
    idcg = sum(1.0 / np.log2(i + 2) for i in range(ideal_n))
    return dcg / idcg if idcg > 0 else 0.0

## Step 4 — 定義五個推薦器 (與 src/recommender.py 同步)

In [ ]:
K = 10

meta_indexed = anime_meta.set_index('anime_id')
popularity_order = (
    meta_indexed.loc[meta_indexed.index.isin(anime_id_to_idx.keys()), 'members']
    .sort_values(ascending=False).index.tolist()
)

def _norm01(x):
    x = np.asarray(x, dtype=np.float32)
    mn, mx = float(x.min()), float(x.max())
    if mx - mn < 1e-9:
        return np.zeros_like(x)
    return (x - mn) / (mx - mn)

def recommend_popularity(uid, k=K):
    seen = user_history.get(uid, set())
    out = [a for a in popularity_order if a not in seen][:k]
    return out

def recommend_content(uid, k=K):
    seen = user_history.get(uid, set())
    seen_idx = [anime_id_to_idx[a] for a in seen if a in anime_id_to_idx]
    if not seen_idx:
        return recommend_popularity(uid, k)
    sims = np.asarray(content_sim[seen_idx].sum(axis=0)).ravel()
    sims[seen_idx] = -np.inf
    top = np.argpartition(-sims, range(min(k*3, len(sims))))[:k*3]
    top = top[np.argsort(-sims[top])][:k]
    return [idx_to_anime_id[i] for i in top]

def recommend_user_cf(uid, k=K):
    if uid not in user_id_to_idx: return recommend_popularity(uid, k)
    u = user_id_to_idx[uid]
    nb_i = user_cf['neighbors_idx'][u]
    nb_s = user_cf['neighbors_sim'][u]
    scores = np.asarray(nb_s @ user_cf['user_item'][nb_i].toarray()).ravel()
    seen = user_history.get(uid, set())
    for a in seen:
        if a in anime_id_to_idx:
            scores[anime_id_to_idx[a]] = -np.inf
    top = np.argpartition(-scores, range(min(k*3, len(scores))))[:k*3]
    top = top[np.argsort(-scores[top])][:k]
    return [idx_to_anime_id[i] for i in top]

def recommend_svd(uid, k=K):
    if uid not in user_id_to_idx: return recommend_popularity(uid, k)
    u = user_id_to_idx[uid]
    scores = (svd['global_mean'] + svd['user_bias'][u] + svd['item_bias']
              + svd['item_factors'] @ svd['user_factors'][u])
    seen = user_history.get(uid, set())
    for a in seen:
        if a in anime_id_to_idx:
            scores[anime_id_to_idx[a]] = -np.inf
    top = np.argpartition(-scores, range(min(k*3, len(scores))))[:k*3]
    top = top[np.argsort(-scores[top])][:k]
    return [idx_to_anime_id[i] for i in top]

def recommend_multimodal(uid, k=K):
    # 期末新增:結合 sentence-transformers 嵌入 (text 模態) + content_sim (structural 模態)
    seen = user_history.get(uid, set())
    seen_idx = [anime_id_to_idx[a] for a in seen if a in anime_id_to_idx]
    if not seen_idx:
        return recommend_popularity(uid, k)
    u_vec = text_emb[seen_idx].mean(axis=0)
    n = np.linalg.norm(u_vec)
    if n > 0: u_vec = u_vec / n
    text_sim = text_emb @ u_vec
    struct_sim = np.asarray(content_sim[seen_idx].sum(axis=0)).ravel().astype(np.float32)
    combined = 0.6 * _norm01(text_sim) + 0.4 * _norm01(struct_sim)
    for a in seen:
        if a in anime_id_to_idx:
            combined[anime_id_to_idx[a]] = -np.inf
    top = np.argpartition(-combined, range(min(k*3, len(combined))))[:k*3]
    top = top[np.argsort(-combined[top])][:k]
    return [idx_to_anime_id[i] for i in top]

RECOMMENDERS = {
    'Popularity (Baseline)': recommend_popularity,
    'Content-Based': recommend_content,
    'User-Based CF': recommend_user_cf,
    'SVD (Matrix Factorization)': recommend_svd,
    'Multimodal (Text+Structural)': recommend_multimodal,  # 期末新增
}

## Step 5 — 跑評估 (5 個模型)

In [ ]:
from tqdm.auto import tqdm

results = {}
for name, fn in RECOMMENDERS.items():
    precs, recs, ndcgs = [], [], []
    for uid in tqdm(eval_users, desc=name):
        rel = gt_dict[uid]
        rec = fn(uid, K)
        precs.append(precision_at_k(rec, rel, K))
        recs.append(recall_at_k(rec, rel, K))
        ndcgs.append(ndcg_at_k(rec, rel, K))
    results[name] = {
        'Precision@10': float(np.mean(precs)),
        'Recall@10': float(np.mean(recs)),
        'NDCG@10': float(np.mean(ndcgs)),
    }

metrics_df = pd.DataFrame(results).T
print(metrics_df.round(4))

## Step 6 — 視覺化比較 (5 個模型並列)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(11, 5))
metrics_df.plot(kind='bar', ax=ax, color=['#1F4E79', '#2E75B6', '#9DC3E6'])
ax.set_title('Model Comparison on Test Set (k=10) — incl. Multimodal')
ax.set_ylabel('score')
ax.set_xticklabels(metrics_df.index, rotation=20, ha='right')
plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/metrics_comparison.png', dpi=120)
plt.show()

## Step 7 — 儲存指標數字 (給 Streamlit 顯示)

In [ ]:
with open(f'{ARTIFACTS_DIR}/metrics.json', 'w') as f:
    json.dump(results, f, indent=2)
print('✅ Saved metrics.json (含 Multimodal)')
!ls -lh {ARTIFACTS_DIR}

## ✅ 完成 — 5 個模型 (Popularity / Content-Based / User-CF / SVD / Multimodal) 都跑完了

把 `metrics.json` 與 `metrics_comparison.png` 重新下載到本地 `artifacts/` 取代舊版即可。
Streamlit 的「模型評估比較」分頁會自動顯示新的 5 行表格 + 5 組條形圖。